# Explore raw telemetry export: 2026-05-30-21-02

Quick exploratory notebook for `../data/raw/2026-05-30-21-02.csv`. It loads the raw telemetry export, identifies rows with missing core telemetry fields, filters to complete rows with `data_schema_version >= 6`, expands JSON metadata fields, checks data quality, and summarizes challenge labels across versions, modes, devices, and difficulty.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

DATA_PATH = Path("../data/raw/2026-05-30-21-02.csv")
assert DATA_PATH.exists(), f"Missing data file: {DATA_PATH.resolve()}"


## Load raw data

In [ ]:
raw_all = pd.read_csv(DATA_PATH)
raw_all["data_schema_version"] = pd.to_numeric(raw_all["data_schema_version"], errors="coerce")
raw = raw_all[raw_all["data_schema_version"].ge(6)].copy()

print(f"Raw rows: {len(raw_all):,} | Filtered rows (data_schema_version >= 6): {len(raw):,} | Columns: {raw.shape[1]:,}")
display(raw.head())
display(raw.dtypes.to_frame("dtype"))


## Parse timestamps and expand `metadata`

In [ ]:
timestamp_cols = ["created_at", "window_started_at", "window_ended_at"]
for col in timestamp_cols:
    raw[col] = pd.to_datetime(raw[col], errors="coerce", utc=True)

def parse_metadata(value):
    if pd.isna(value):
        return {}
    if isinstance(value, dict):
        parsed = value
    else:
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            return {}
    if not isinstance(parsed, dict):
        return {}

    # Keep all metadata fields except the nested failed_jump_counts dictionary.
    return {key: val for key, val in parsed.items() if key != "failed_jump_counts"}

core_cols = [
    "event_type",
    "game_version",
    "game_mode",
    "challenge_label",
    "device_type",
    "deployment_context",
    "data_schema_version",
]
core_cols = [col for col in core_cols if col in raw.columns]

# Preserve the pre-fix view so the NaN-containing rows are visible for diagnosis.
metadata_unaligned = pd.json_normalize(raw["metadata"].map(parse_metadata)).add_prefix("meta_")
df_unaligned = pd.concat([raw.drop(columns=["metadata"]), metadata_unaligned], axis=1)
nan_rows = df_unaligned[df_unaligned[core_cols].isna().any(axis=1)].copy()

print(f"Rows with NaN in core fields before cleanup: {len(nan_rows):,}")
display(nan_rows.head())
display(nan_rows[core_cols].isna().sum().to_frame("nan_rows"))

# Reset the filtered raw index before concatenating normalized metadata to avoid index-alignment NaNs.
raw = raw[raw[core_cols].notna().all(axis=1)].reset_index(drop=True)
metadata = pd.json_normalize(raw["metadata"].map(parse_metadata)).add_prefix("meta_")
df = pd.concat([raw.drop(columns=["metadata"]), metadata], axis=1)

# Keep the analysis frame restricted to complete core telemetry rows.
df = df[df[core_cols].notna().all(axis=1)].copy()

# Parse any metadata timestamps after expansion.
for col in [c for c in df.columns if c.endswith("_at") or c == "meta_logged_at"]:
    if df[col].dtype == "object":
        parsed = pd.to_datetime(df[col], errors="coerce", utc=True)
        if parsed.notna().any():
            df[col] = parsed

print(f"Expanded rows after NaN cleanup: {len(df):,} | Expanded columns: {df.shape[1]:,}")
display(df.head())


## Data quality checks

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
    "unique": df.nunique(dropna=True),
}).sort_values(["missing_pct", "unique"], ascending=[False, True])

display(quality)
print("Duplicate ids:", df["id"].duplicated().sum() if "id" in df else "n/a")
print("Date range:", df["created_at"].min(), "to", df["created_at"].max())


## Dataset composition

In [ ]:
for col in ["event_type", "game_version", "game_mode", "challenge_label", "device_type", "deployment_context", "data_schema_version"]:
    if col in df:
        print(f"\n{col}")
        display(df[col].value_counts(dropna=False).to_frame("rows"))


## Telemetry windows only

In [ ]:
windows = df[df["event_type"].eq("telemetry_window")].copy()
print(f"Telemetry windows: {len(windows):,}")

summary_cols = [
    "metric_value", "difficulty", "score", "height_climbed", "window_index",
    "meta_deaths", "meta_new_platforms_reached", "meta_failed_jump_attempts",
    "meta_jump_key_presses", "meta_left_key_presses", "meta_right_key_presses",
    "meta_total_horizontal_movement_px", "meta_seconds_since_flag",
]
summary_cols = [c for c in summary_cols if c in windows]
display(windows[summary_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)


## Data entries by difficulty


In [ ]:
difficulty_values = list(range(1, 11))
difficulty_counts = (
    windows["difficulty"]
    .value_counts(dropna=False)
    .reindex(difficulty_values, fill_value=0)
    .rename_axis("difficulty")
    .to_frame("rows")
)
display(difficulty_counts)

difficulty_view_cols = [
    "id", "created_at", "session_id", "game_mode", "challenge_label",
    "device_type", "difficulty", "score", "height_climbed", "window_index",
    "meta_deaths", "meta_failed_jump_attempts", "meta_new_platforms_reached",
]
difficulty_view_cols = [col for col in difficulty_view_cols if col in windows.columns]

for difficulty in difficulty_values:
    difficulty_rows = windows[windows["difficulty"].eq(difficulty)].sort_values("created_at")
    print(f"\nDifficulty {difficulty}: {len(difficulty_rows):,} rows")
    display(difficulty_rows[difficulty_view_cols].head(20))


## Challenge labels by key slices

In [ ]:
def label_crosstab(index_col):
    if index_col not in windows:
        return None
    table = pd.crosstab(windows[index_col], windows["challenge_label"], margins=True)
    share = pd.crosstab(windows[index_col], windows["challenge_label"], normalize="index").round(3)
    display(table)
    display(share)

for col in ["game_version", "game_mode", "device_type", "deployment_context", "difficulty", "data_schema_version"]:
    print(f"\nchallenge_label by {col}")
    label_crosstab(col)


## Session-level view

In [ ]:
session_summary = (
    windows.groupby("session_id", dropna=False)
    .agg(
        windows=("id", "count"),
        started_at=("created_at", "min"),
        ended_at=("created_at", "max"),
        game_version=("game_version", lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
        game_mode=("game_mode", lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
        device_type=("device_type", lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
        max_score=("score", "max"),
        max_height=("height_climbed", "max"),
        mean_difficulty=("difficulty", "mean"),
        deaths=("meta_deaths", "sum"),
        failed_jumps=("meta_failed_jump_attempts", "sum"),
        platforms_reached=("meta_new_platforms_reached", "sum"),
    )
    .sort_values(["windows", "max_score"], ascending=False)
)

display(session_summary.head(20))
display(session_summary.describe(include="all"))


## Correlations and quick plots

In [ ]:
numeric_cols = windows.select_dtypes(include=["number"]).columns
focus_cols = [c for c in [
    "metric_value", "difficulty", "score", "height_climbed", "window_index",
    "meta_deaths", "meta_new_platforms_reached", "meta_failed_jump_attempts",
    "meta_jump_key_presses", "meta_total_horizontal_movement_px", "meta_seconds_since_flag",
] if c in numeric_cols]

display(windows[focus_cols].corr(numeric_only=True).round(2))

try:
    import matplotlib.pyplot as plt

    axes = windows[focus_cols].hist(figsize=(14, 10), bins=30)
    plt.tight_layout()
    plt.show()

    windows.sort_values("created_at").plot(
        x="created_at", y=[c for c in ["difficulty", "score", "height_climbed"] if c in windows],
        figsize=(14, 5), alpha=0.75
    )
    plt.show()
except ImportError:
    print("matplotlib is not installed in this environment; skipping plots.")


## Candidate modeling frame

In [ ]:
feature_candidates = [
    "difficulty", "score", "height_climbed", "window_index",
    "meta_deaths", "meta_new_platforms_reached", "meta_failed_jump_attempts",
    "meta_distinct_failed_jumps", "meta_repeated_failed_jump_attempts",
    "meta_jump_key_presses", "meta_left_key_presses", "meta_right_key_presses",
    "meta_total_horizontal_movement_px", "meta_seconds_since_flag",
    "meta_platform_gap_y_avg_px", "meta_platform_width_avg_px",
    "meta_platform_x_shift_min_px", "meta_platform_x_shift_max_px",
    "meta_platform_speed_px_per_frame",
]
feature_candidates = [c for c in feature_candidates if c in windows]
model_frame = windows[feature_candidates + ["challenge_label"]].dropna(subset=["challenge_label"]).copy()

print(model_frame.shape)
display(model_frame.head())
display(model_frame["challenge_label"].value_counts(normalize=True).to_frame("share"))
